In [1]:
# 0. Environment Setup

# Clone the gemma repository
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository for UI/UX
!git clone https://github.com/google-deepmind/dialog.git || true
!pip install -q ./dialog

# Ensure local modules are in path
import sys
import os
sys.path.append(os.getcwd())

Cloning into 'gemma'...
remote: Enumerating objects: 2488, done.
remote: Counting objects: 100% (1139/1139), done.
remote: Compressing objects: 100% (362/362), done.
remote: Total 2488 (delta 927), reused 778 (delta 777), pack-reused 1349 (from 3)
Receiving objects: 100% (2488/2488), 1.19 MiB | 3.94 MiB/s, done.
Resolving deltas: 100% (1754/1754), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 76.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.4/318.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.8/633.8 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.6/619.6 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 19.1 MB/s 

In [2]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 444, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 444 (delta 51), reused 50 (delta 50), pack-reused 391 (from 1)
Receiving objects: 100% (444/444), 108.60 MiB | 24.02 MiB/s, done.
Resolving deltas: 100% (277/277), done.
/content/Titans_jax


In [3]:
import importlib
import jax
import os
import gemma_titans
# importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils
from gemma import gm

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"

# jax.config.update("jax_disable_jit", True) # Temporarily disable JIT to bypass the hashing error

JAX Backend: gpu
Devices: [CudaDevice(id=0)]


In [9]:
import dataclasses
inference_config = Gemma_Titans_Config(
    **{f.name: getattr(Gemma3_1B_Titans.config, f.name) for f in dataclasses.fields(Gemma_Titans_Config) if f.name != 'is_training_mode'},
    is_training_mode=False # <--- ВЫКЛЮЧАЕМ ТРЕНИРОВКУ ДЛЯ СЭМПЛЕРА
)

In [10]:
import google.colab
import shutil
import orbax.checkpoint as ocp

if os.path.exists('/content/Titans_jax/saved_titans_delta.zip'):
    shutil.unpack_archive('/content/Titans_jax/saved_titans_delta.zip',"./saved_titans_delta")

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(load_path)

import jax.numpy as jnp
# gemma_config = Gemma3_1B_Titans.config
# inference_config = Gemma_Titans_Config(
#     **{f.name: getattr(Gemma3_1B_Titans.config, f.name) for f in dataclasses.fields(_config.TransformerConfig)},
#     is_training_mode=False # <--- ВЫКЛЮЧАЕМ ТРЕНИРОВКУ ДЛЯ СЭМПЛЕРА
# )

# gemma_config.is_training_mode = False

model = Gemma3_1B_Titans(
    config=inference_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
)

print("Loading weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
loaded_titans_params = load_titans_weights("./saved_titans_delta")
merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

tokenizer = gm.text.Gemma3Tokenizer()

Loading weights...


## 4. Interactive Dialogue with `google-deepmind/dialog`

In [12]:
import dialog
from gemma import gm
import jax

# Initialize Sampler and Conversation
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

conv = dialog.Conversation()


def chat(user_input: str):
    global conv
    # Add user message
    conv += dialog.User(user_input)

    # Convert conversation to prompt (Gemma 3 format)
    prompt = conv.as_text(training=False)

    # Generate response
    # Note: Sampler handles the caching of Titans memory automatically
    response_text = sampler.sample(prompt, max_new_tokens=128)

    # Add model response to UI
    conv += dialog.Model(response_text)
    conv.show()

# Example usage:
# chat("Привет! Кто такие Титаны в мифологии?")

In [ ]:
chat("Who is Titans in the anchient greek mythology?")
